<a href="https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
I chose a Decision Tree because it can model non-linear relationships while remaining relatively easy to interpret. This is useful for this task because I want to understand which observable signals are associated with the pages the model prioritizes. I will compare the model against my Week-4 baseline using the same evaluation metric and the same test data. I will not treat greater model complexity as automatically better.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
I will use a train/test split so the model is evaluated on data it did not train on. I will use 80% of the rows for training and 20% for testing. The split will be stratified by the target label so the proportion of declining and non-declining pages remains similar in both sets. The Week-4 baseline will be evaluated on the same test rows for a fair comparison.


In [7]:
import os
import subprocess

REPO_URL = "https://github.com/ubaid8878/Flyrank-ML-Internship"
REPO_DIR = "/content/Flyrank-ML-Internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())
print("Dataset exists:", os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
))

Current folder: /content/Flyrank-ML-Internship
Dataset exists: True


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded:", df.shape)

# Target
y = df["trend_direction"].str.lower().eq("down").astype(int)

# Observable features only
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Create feature matrix
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# 80% training / 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training declining rate:", round(y_train.mean(), 3))
print("Testing declining rate:", round(y_test.mean(), 3))

Dataset loaded: (30000, 44)
Training rows: 24000
Testing rows: 6000
Training declining rate: 0.542
Testing declining rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
I will train a Decision Tree using the six observable features defined in Section 2. I will evaluate the model on the 20% test set that was not used during training. For a fair comparison, I will also evaluate my Week-4 baseline on exactly the same test rows. I will use Precision@50 because the Week-4 baseline was designed to rank pages for action, where the highest-ranked pages are the most important.


In [9]:
from sklearn.tree import DecisionTreeClassifier

# Train the Decision Tree on the training set only
model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# Model scores for the test set
model_scores = model.predict_proba(X_test)[:, 1]

# Precision@K function
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Model performance
model_p20 = precision_at_k(model_scores, y_test, 20)
model_p50 = precision_at_k(model_scores, y_test, 50)

print("Decision Tree Precision@20:", round(model_p20, 3))
print("Decision Tree Precision@50:", round(model_p50, 3))

Decision Tree Precision@20: 0.65
Decision Tree Precision@50: 0.68


In [10]:
# Evaluate the Week-4 baseline on the SAME test rows

baseline_test_scores = (
    (df.loc[X_test.index, "days_since_last_update"] >= 180).astype(int) * 2
    + (df.loc[X_test.index, "ctr"] < df["ctr"].median()).astype(int)
)

baseline_p20 = precision_at_k(baseline_test_scores, y_test, 20)
baseline_p50 = precision_at_k(baseline_test_scores, y_test, 50)

print("Week-4 Baseline Precision@20:", round(baseline_p20, 3))
print("Week-4 Baseline Precision@50:", round(baseline_p50, 3))

Week-4 Baseline Precision@20: 0.5
Week-4 Baseline Precision@50: 0.44


In [11]:
comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Decision Tree"],
    "Precision@20": [baseline_p20, model_p20],
    "Precision@50": [baseline_p50, model_p50]
})

print(comparison.round(3).to_string(index=False))

         Method  Precision@20  Precision@50
Week-4 Baseline          0.50          0.44
  Decision Tree          0.65          0.68


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
The Decision Tree performed better than my Week-4 baseline on the same test set. Precision@20 increased from 0.50 for the baseline to 0.65 for the Decision Tree. Precision@50 increased from 0.44 to 0.68.

This is an observed improvement on this test split, not proof that the model will always perform better. The model can combine multiple observable signals instead of relying on the single baseline rule.

Some errors are expected because pages with similar observable signals can have different outcomes. Low impression volume can also make CTR less stable, so some pages may be prioritized even when the signal is noisy. The model should therefore be treated as decision-support rather than an automatic decision maker.


In [12]:
# Feature importance from the Decision Tree

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("Decision Tree feature importance:")
print(importance.round(4).to_string(index=False))

Decision Tree feature importance:
               feature  importance
       impressions_90d      0.5661
      content_age_days      0.2687
          avg_position      0.0904
                   ctr      0.0748
days_since_last_update      0.0000
            word_count      0.0000


In [13]:
# Inspect test examples where the model's prediction disagrees with the actual label

test_results = df.loc[X_test.index, [
    "content_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction"
]].copy()

test_results["actual"] = y_test
test_results["model_score"] = model_scores
test_results["predicted"] = (model_scores >= 0.5).astype(int)

errors = test_results[
    test_results["actual"] != test_results["predicted"]
].copy()

print("Total test errors:", len(errors))
print("\nSample errors:")
print(errors.head(10).to_string(index=False))

Total test errors: 2210

Sample errors:
          content_id  content_age_days  days_since_last_update  impressions_90d  avg_position  ctr  word_count trend_direction  actual  model_score  predicted
content_a128777972dd               557                      20             1666           7.9 0.00         NaN            down       1     0.438221          0
content_a1f8fe3ed192               223                     104              511          15.0 0.39      6466.0          stable       0     0.509929          1
content_b6adeae99103               463                      22              881          15.8 0.11         NaN            down       1     0.438221          0
content_a0744f34c5d8               358                     104               31          12.0 0.00      1204.0            down       1     0.438221          0
content_4ef406aa3516               175                      20              127          18.3 0.00      2892.0             new       0     0.639099          1
conten

In [ ]:
Feature interpretation:

The feature importance output shows which observable signals the tree relied on most in this run. These are associations used by the model, not proof of causation. The most important feature should be interpreted as a useful signal for prioritization, while recognizing that other factors may explain the outcome.

Error interpretation:

The model still makes errors on the test set. These errors show that the available features do not perfectly separate declining from non-declining pages. In particular, noisy CTR estimates and differences between pages with similar observable signals can lead to incorrect prioritization.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.